In [1]:
from typing import Literal 
from pydantic import BaseModel, Field 
 
# Schema for Sentiment Analysis 
class SentimentSchema(BaseModel): 
   sentiment: Literal["positive", "negative"] = Field( 
       description="Sentiment of the review" 
   ) 
 


In [3]:
# Schema for Deep Diagnostic Evaluation on Negative Reviews 
class DiagnosisSchema(BaseModel): 
   issue_type: Literal["ux", "performance", "bug", "support", "other"] = Field( 
       description="Category of issue mentioned in the review" 
   ) 
   tone: Literal["frustrated", "angry", "neutral", "disappointed"] = Field( 
       description="Emotional tone expressed by the user" 
   ) 
   urgency: Literal["low", "medium", "high"] = Field( 
       description="How urgent or critical the issue appears to be" 
   )

In [4]:
from typing import TypedDict, Optional, Dict, Any 
 
class ReviewState(TypedDict): 
   review: str 
   sentiment: Optional[str] 
   diagnosis: Optional[Dict[str, Any]] 
   response: Optional[str] 
 

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="grok-4.6",
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1",
)
structured_llm_sentiment = llm.with_structured_output(SentimentSchema) 
structured_llm_diagnosis = llm.with_structured_output(DiagnosisSchema) 
 
# Node 1: Sentiment Classification 
def find_sentiment(state: ReviewState): 
   prompt = f"For the following review, find out the sentiment: {state['review']}" 
   res = structured_llm_sentiment.invoke(prompt) 
   return {"sentiment": res.sentiment} 
 
# Conditional Router Function 
def check_sentiment(state: ReviewState) -> str: 
   if state["sentiment"] == "positive": 
       return "positive_response" 
   else: 
       return "run_diagnosis" 
 
# Node 2A: Positive Path Node 
def positive_response(state: ReviewState): 
   prompt = f"Write a warm thank-you message responding to this review: {state['review']}. Ask for web feedback." 
   res = llm.invoke(prompt) 
   return {"response": res.content} 
 
# Node 2B: Negative Path Diagnostic Node 
def run_diagnosis(state: ReviewState): 
   prompt = f"Diagnose this negative review: {state['review']}" 
   res = structured_llm_diagnosis.invoke(prompt) 
   return {"diagnosis": res.model_dump()} 
 
# Node 3B: Resolution Generator Node 
def negative_response(state: ReviewState): 
   diag = state["diagnosis"] 
   prompt = f""" 
   You are a support assistant. User faced a {diag['issue_type']} issue, 
   sounded {diag['tone']}, and marked urgency as {diag['urgency']}. 
   Write an empathetic, helpful resolution message. 
   """ 
   res = llm.invoke(prompt) 
   return {"response": res.content} 
 



workflow = StateGraph(ReviewState) 
 
workflow.add_node("find_sentiment", find_sentiment) 
workflow.add_node("positive_response", positive_response) 
workflow.add_node("run_diagnosis", run_diagnosis) 
workflow.add_node("negative_response", negative_response) 
 
workflow.add_edge(START, "find_sentiment") 
 
# Conditional Router Edge 
workflow.add_conditional_edges("find_sentiment", check_sentiment) 
 
# Edge connections to END 
workflow.add_edge("positive_response", END) 
workflow.add_edge("run_diagnosis", "negative_response") 
workflow.add_edge("negative_response", END) 
 
app = workflow.compile() 
 

ModuleNotFoundError: No module named 'langchain_openai'